In [ ]:
!python -m pip install --user tensorflow opencv-python matplotlib seaborn numpy pillow

In [ ]:
!python -m pip install --user ipykernel
!python -m ipykernel install --user --name tf_env --display-name "TensorFlow Env"


In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)


In [ ]:
!python -m pip install --user ipykernel
!python -m ipykernel install --user --name tf_env --display-name "TensorFlow Env"


In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("TensorFlow:", tf.__version__)


In [ ]:
# STEP 3: Set Project Paths and Basic Settings

# BASE_DIR = parent folder of "notebooks"
import os

BASE_DIR = os.path.abspath("..")
print("BASE_DIR:", BASE_DIR)

# Image settings
IMG_SIZE = 224
BATCH_SIZE = 16

# Dataset directories
DATA_DIR = os.path.join(BASE_DIR, "dataset")

train_dir = os.path.join(DATA_DIR, "train")
val_dir   = os.path.join(DATA_DIR, "val")
test_dir  = os.path.join(DATA_DIR, "test")

print("Train directory:", train_dir)
print("Validation directory:", val_dir)
print("Test directory:", test_dir)


In [ ]:
# STEP 4: Data Generators (Training, Validation, Testing)

# Training data augmentation + normalization
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation & Test data (only normalization)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Training generator
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

# Validation generator
val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

# Test generator
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# Print class mapping
print("Class indices:", train_gen.class_indices)


In [ ]:
# STEP 5: Build MobileNetV2 Model for 22-class Skin Disease Classification

# Load MobileNetV2 base model (without top layer)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze base model layers (so they don’t train)
base_model.trainable = False

# Add custom layers on top
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.2)(x)

# num_classes = 22 (automatically detected from train_gen)
num_classes = train_gen.num_classes

output_layer = Dense(num_classes, activation="softmax")(x)

# Create final model
model = Model(inputs=base_model.input, outputs=output_layer)

# Compile model
model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=1e-4),
    metrics=["accuracy"]
)

# Show model structure
model.summary()


In [ ]:
# STEP 6: Train the Model

# Create models directory if not exists
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Save best model only
checkpoint_path = os.path.join(MODELS_DIR, "skin_detector_best.h5")

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

EPOCHS = 20   # You can increase later if needed

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)


In [ ]:
# STEP 7: Plot Training Accuracy & Loss

import matplotlib.pyplot as plt

# Accuracy Plot
plt.figure(figsize=(10,4))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

# Loss Plot
plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import os

# Project root (parent of notebooks folder)
BASE_DIR = os.path.abspath("..")
print("BASE_DIR:", BASE_DIR)

# Paths
DATA_DIR = os.path.join(BASE_DIR, "dataset")
MODELS_DIR = os.path.join(BASE_DIR, "models")

# Settings
IMG_SIZE = 224
BATCH_SIZE = 16


In [ ]:
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Paths
BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "dataset")
MODELS_DIR = os.path.join(BASE_DIR, "models")
test_dir = os.path.join(DATA_DIR, "test")

# Recreate test generator
test_datagen = ImageDataGenerator(rescale=1./255)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# Load model
best_model_path = os.path.join(MODELS_DIR, "skin_detector_best.h5")
best_model = load_model(best_model_path)

# Evaluate
test_loss, test_acc = best_model.evaluate(test_gen)

print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
# True labels
y_true = test_gen.classes

# Model predictions
y_pred_prob = best_model.predict(test_gen)

# Convert probabilities to class labels
y_pred = np.argmax(y_pred_prob, axis=1)


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=test_gen.class_indices.keys(),
    yticklabels=test_gen.class_indices.keys(),
    square=True
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image


In [ ]:
# Load trained model
best_model_path = os.path.join(MODELS_DIR, "skin_detector_best.h5")
model = load_model(best_model_path)

# 🔥 Manually define class names (same as dataset folders)
class_names = [
    'Acne', 'Actinic_Keratosis', 'Benign_tumors', 'Bullous',
    'Candidiasis', 'DrugEruption', 'Eczema', 'Infestations_Bites',
    'Lichen', 'Lupus', 'Moles', 'Psoriasis', 'Rosacea',
    'Seborrh_Keratoses', 'SkinCancer', 'Sun_Sunlight_Damage',
    'Tinea', 'Unknown_Normal', 'Vascular_Tumors',
    'Vasculitis', 'Vitiligo', 'Warts'
]

print("Total Classes:", len(class_names))
print("Classes:", class_names)


In [ ]:
# 👇 APNI IMAGE KA PATH DALO
img_path = r"C:\Users\user\Downloads\WhatsApp Image 2025-12-13 at 2.53.27 PM.jpeg"


# Load & preprocess image
img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
predictions = model.predict(img_array)
predicted_class = np.argmax(predictions)
confidence = np.max(predictions)

# Output
print("Predicted Disease:", class_names[predicted_class])
print(f"Confidence: {confidence * 100:.2f}%")

# Show image
plt.imshow(img)
plt.axis("off")
plt.title(f"{class_names[predicted_class]} ({confidence*100:.2f}%)")
plt.show()


In [ ]:
import os
import shutil
import random

# ===== CONFIG =====
SOURCE_DATASET = r"C:\Users\user\Desktop\Skin Detector\archive\dataset"
TARGET_DATASET = r"C:\Users\user\Desktop\Skin Detector\dataset_7class"
BACKUP_FOLDER  = r"C:\Users\user\Desktop\Skin Detector\backup_removed"

SELECTED_CLASSES = [
    "Acne",
    "Eczema",
    "Psoriasis",
    "Warts",
    "Vitiligo",
    "Rosacea",
    "Normal"
]

IMAGES_PER_CLASS = 215
TRAIN_SPLIT = 150
VAL_SPLIT   = 32
TEST_SPLIT  = 33

random.seed(42)

# ==================

def make_dirs():
    for split in ["train", "val", "test"]:
        for cls in SELECTED_CLASSES:
            os.makedirs(os.path.join(TARGET_DATASET, split, cls), exist_ok=True)
    os.makedirs(BACKUP_FOLDER, exist_ok=True)

make_dirs()

for cls in SELECTED_CLASSES:
    print(f"\nProcessing class: {cls}")
    
    # collect images from all splits
    images = []
    for split in ["train", "val", "test"]:
        src = os.path.join(SOURCE_DATASET, split, cls)
        if os.path.exists(src):
            for f in os.listdir(src):
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    images.append(os.path.join(src, f))
    
    random.shuffle(images)
    
    selected = images[:IMAGES_PER_CLASS]
    removed  = images[IMAGES_PER_CLASS:]
    
    # copy selected images
    for i, img in enumerate(selected):
        if i < TRAIN_SPLIT:
            dst = os.path.join(TARGET_DATASET, "train", cls)
        elif i < TRAIN_SPLIT + VAL_SPLIT:
            dst = os.path.join(TARGET_DATASET, "val", cls)
        else:
            dst = os.path.join(TARGET_DATASET, "test", cls)
        shutil.copy(img, dst)
    
    # move removed images to backup
    backup_cls = os.path.join(BACKUP_FOLDER, cls)
    os.makedirs(backup_cls, exist_ok=True)
    for img in removed:
        shutil.move(img, backup_cls)

    print(f"  Selected: {len(selected)} | Moved to backup: {len(removed)}")

print("\n✅ Dataset successfully reduced & split safely!")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH = 16

SPLIT_DIR = r"C:\Users\user\Desktop\Skin Detector\archive\dataset_split"

# Train generator (with augmentation)
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation & Test generator (NO augmentation)
val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    SPLIT_DIR + "/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

val_data = val_test_gen.flow_from_directory(
    SPLIT_DIR + "/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

test_data = val_test_gen.flow_from_directory(
    SPLIT_DIR + "/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH = 16

SPLIT_DIR = r"C:\Users\user\Desktop\Skin Detector\archive\dataset_split"

# Train generator (with augmentation)
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation & Test generator (NO augmentation)
val_test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    SPLIT_DIR + "/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

val_data = val_test_gen.flow_from_directory(
    SPLIT_DIR + "/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical"
)

test_data = val_test_gen.flow_from_directory(
    SPLIT_DIR + "/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)
